In [17]:
# 1. 导入所有依赖库
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2" 
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np
import random

# 导入我们的核心模块
from models.apm_former import APM_Former_ImageOnly
from utils.dataset import get_adni_dataloaders
# 修正导入：只导入config里存在的变量
from utils.config import TRAIN_CSV, VAL_CSV, BATCH_SIZE, NUM_WORKERS, TRAIN_IMG_SIZE
from utils.logging_utils import get_logger
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR


# 日志
logger = get_logger("Train")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # torch.cuda.manual_seed_all(seed)  # 如果用多卡
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False



# seed42目前效果最好，75%
# seed 3407

In [18]:
# 2. 超参数配置
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LEARNING_RATE = 3e-4
# warmup_epochs = 10  # 前 5 个 epoch 用来预热
WEIGHT_DECAY = 1e-5
NUM_EPOCHS = 50
GRADIENT_ACCUMULATION_STEPS = 1
NUM_CLASSES = 2
FEATURE_SIZE = 48
GUIDE_CHANNELS = 18
SAVE_PATH = "checkpoints/best_model.pth"
PRETRAINED_SWIN_PATH = "checkpoints/model_swinvit.pt"
SEED = 3407

set_seed(SEED)

logger.info("=" * 50)
logger.info("🌟 本次实验配置档案 (Experiment Config) 🌟")
logger.info("=" * 50)
logger.info(f"▶ 随机种子 (SEED): {SEED}")
logger.info(f"▶ 峰值学习率 (LR): {LEARNING_RATE}")
# logger.info(f"▶ 预热轮数 (Warmup Epochs): {warmup_epochs}")
logger.info(f"▶ 批次大小 (Batch Size): {BATCH_SIZE}")
logger.info(f"▶ 优化器: AdamW (Weight Decay: {WEIGHT_DECAY})")
logger.info(f"▶ 架构说明: 启用浅层特征多尺度融合 (shallow_downsample)")
logger.info(f"▶ 架构说明: 纯空间对齐，未启用 m_k 调制掩码")
logger.info("=" * 50)

logger.info(f"训练设备: {DEVICE}")
logger.info(f"批次大小: {BATCH_SIZE}")
logger.info(f"总轮数: {NUM_EPOCHS}")

2026-05-07 03:31:40,805 - Train - INFO - ==================================================
2026-05-07 03:31:40,807 - Train - INFO - 🌟 本次实验配置档案 (Experiment Config) 🌟
2026-05-07 03:31:40,808 - Train - INFO - ==================================================
2026-05-07 03:31:40,809 - Train - INFO - ▶ 随机种子 (SEED): 3407
2026-05-07 03:31:40,811 - Train - INFO - ▶ 峰值学习率 (LR): 0.0003
2026-05-07 03:31:40,813 - Train - INFO - ▶ 批次大小 (Batch Size): 4
2026-05-07 03:31:40,814 - Train - INFO - ▶ 优化器: AdamW (Weight Decay: 1e-05)
2026-05-07 03:31:40,816 - Train - INFO - ▶ 架构说明: 启用浅层特征多尺度融合 (shallow_downsample)
2026-05-07 03:31:40,817 - Train - INFO - ▶ 架构说明: 纯空间对齐，未启用 m_k 调制掩码
2026-05-07 03:31:40,819 - Train - INFO - ==================================================
2026-05-07 03:31:40,821 - Train - INFO - 训练设备: cuda
2026-05-07 03:31:40,822 - Train - INFO - 批次大小: 4
2026-05-07 03:31:40,823 - Train - INFO - 总轮数: 50


In [19]:
# 3. 加载训练集 + 验证集 DataLoader
logger.info("正在加载数据集...")
train_loader, val_loader = get_adni_dataloaders(
    train_csv=TRAIN_CSV,
    val_csv=VAL_CSV,
    batch_size=BATCH_SIZE,
    target_size=TRAIN_IMG_SIZE,
    num_workers=NUM_WORKERS
)

2026-05-07 03:31:40,888 - Train - INFO - 正在加载数据集...
2026-05-07 03:31:40,935 - utils.dataset - INFO - ✅ 成功加载 659 个样本
2026-05-07 03:31:40,942 - utils.dataset - INFO - ✅ 成功加载 82 个样本
2026-05-07 03:31:40,943 - utils.dataset - INFO - ✅ DataLoader 构建完成！
2026-05-07 03:31:40,944 - utils.dataset - INFO - 训练集批次: 165，验证集批次: 21


In [20]:
# 4. 初始化模型（完整版！）
logger.info("初始化 APM-Former 模型...")
model = APM_Former_ImageOnly(
    img_size=TRAIN_IMG_SIZE,
    in_channels=1,
    num_classes=NUM_CLASSES,
    feature_size=FEATURE_SIZE,
    guide_channels=GUIDE_CHANNELS,
    pretrained_swin_path=PRETRAINED_SWIN_PATH # 传入刚刚下载的权重
).to(DEVICE)

# 打印模型参数量
total_params = sum(p.numel() for p in model.parameters())
logger.info(f"模型总参数量: {total_params / 1e6:.2f} M")

2026-05-07 03:31:40,991 - Train - INFO - 初始化 APM-Former 模型...
/home/ubuntu/Code/APM_Former_Project/models/apm_former.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  che

加载SwinUNETR预训练权重：checkpoints/model_swinvit.pt


2026-05-07 03:31:42,524 - Train - INFO - 模型总参数量: 64.39 M


In [21]:
# 5. 损失函数 + 两阶段优化器设置
from monai.losses import FocalLoss
from torch.optim.lr_scheduler import CosineAnnealingLR

# ==========================================
# 损失函数 (保持你原来的完美配置不变)
# ==========================================
weights = torch.tensor([1.0, 1.768]).to(DEVICE) 
criterion = FocalLoss(weight=weights, gamma=2.0, to_onehot_y=True).to(DEVICE)

# ==========================================
# 两阶段训练：阶段一 (冻结主干)
# ==========================================
UNFREEZE_EPOCH = 5  # 设定前 5 个 epoch 为阶段一

# 1. 冻结 Swin 主干网络
logger.info("🔒 阶段一：冻结 Swin 主干网络，仅训练新增模块...")
for name, param in model.named_parameters():
    if "swin_backbone" in name:
        param.requires_grad = False
    else:
        param.requires_grad = True

# 2. 定义优化器：极其重要！这里加了 filter，只把没冻结的参数交给优化器
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), 
    lr=3e-4, 
    weight_decay=WEIGHT_DECAY
)

# 3. 阶段一的调度器：不需要预热，直接用 5 个 epoch 的余弦退火
scheduler = CosineAnnealingLR(optimizer, T_max=UNFREEZE_EPOCH)

2026-05-07 03:31:42,572 - Train - INFO - 🔒 阶段一：冻结 Swin 主干网络，仅训练新增模块...


In [22]:
# 6. 训练/验证函数
def train_epoch(model, loader, criterion, optimizer, device, accum_steps):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    optimizer.zero_grad()
    for idx, (images, labels) in enumerate(tqdm(loader, desc="训练中")):
        images = images.to(device)
        labels = labels.to(device)

        logits, _, _ = model(images)
        loss = criterion(logits, labels.unsqueeze(1))
        loss = loss / accum_steps

        loss.backward()
        # 👇 就在 optimizer.step() 之前，加入这行“梯度裁剪”代码！
        # max_norm=1.0 或 2.0 是最常用的安全值
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        if (idx + 1) % accum_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accum_steps
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = 100 * correct / total
    return avg_loss, acc

@torch.no_grad()
def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="验证中"):
        images = images.to(device)
        labels = labels.to(device)

        logits, _, _ = model(images)
        loss = criterion(logits, labels.unsqueeze(1))

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = 100 * correct / total
    return avg_loss, acc

In [23]:
# # %% 排查：查看训练集、验证集的标签分布
# import pandas as pd
# from utils.config import TRAIN_CSV, VAL_CSV

# train_df = pd.read_csv(TRAIN_CSV)
# val_df = pd.read_csv(VAL_CSV)

# print("===== 训练集标签分布 =====")
# print(train_df['Label'].value_counts())
# print("比例：", train_df['Label'].value_counts(normalize=True))

# print("\n===== 验证集标签分布 =====")
# print(val_df['Label'].value_counts())
# print("比例：", val_df['Label'].value_counts(normalize=True))

In [24]:
# %% 排查：查看 DataLoader 中的标签统计
# from collections import Counter

# train_labels = []
# for imgs, lbls in train_loader:
#     train_labels.extend(lbls.numpy().tolist())

# print("训练集加载到的标签统计：", Counter(train_labels))

In [25]:
# # 强制给模型看标签1，看是否输出正常
# model.eval()
# test_img = torch.randn(1,1,96,96,96).to(DEVICE)
# with torch.no_grad():
#     logits,_,_ = model(test_img)
#     print("模型输出概率：", torch.softmax(logits, dim=1))

In [26]:
# # %% 训练1个batch，打印梯度！！！
# model.train()
# images, labels = next(iter(train_loader))
# images = images.to(DEVICE)
# labels = labels.to(DEVICE)

# optimizer.zero_grad()
# logits, _, _ = model(images)
# loss = criterion(logits, labels)
# loss.backward()

# # 打印Swin主干的梯度（如果是None/0，就是主干冻结！）
# print("Loss:", loss.item())
# print("Swin主干参数梯度：")
# for name, param in model.swin_backbone.named_parameters():
#     if param.grad is not None:
#         print(f"{name}: 梯度均值 = {param.grad.mean().item()}")
#         break
#     else:
#         print(f"{name}: 梯度为 None ❌（冻结了！）")
#         break

In [27]:
# 7. 主训练循环
best_val_acc = 0.0
logger.info("=" * 50)
logger.info("开始训练！")
logger.info("=" * 50)

for epoch in range(NUM_EPOCHS):
    # 👇 ================= 新增：解冻机关 ================= 👇
    if epoch == UNFREEZE_EPOCH:
        logger.info("\n" + "🚀" * 20)
        logger.info("🔓 阶段二触发：解冻 Swin 主干网络，开始全员微调！")
        logger.info("🚀" * 20)
        
        # 将所有参数解冻
        for param in model.parameters():
            param.requires_grad = True
            
        # 重新定义优化器（包含主干网络），并大幅降低学习率来保护老专家！
        # 这里用 1e-4 或 5e-5 进行微调最合适
        optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=WEIGHT_DECAY)
        
        # 重新定义剩下 epoch 的调度器
        scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - UNFREEZE_EPOCH)
    # 👆 =================================================== 👆
    logger.info(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE, GRADIENT_ACCUMULATION_STEPS)
    val_loss, val_acc = val_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step()

    logger.info(f"训练损失: {train_loss:.4f} | 训练准确率: {train_acc:.2f}%")
    logger.info(f"验证损失: {val_loss:.4f} | 验证准确率: {val_acc:.2f}%")
    logger.info(f"当前学习率: {optimizer.param_groups[0]['lr']:.8f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        logger.info(f"✅ 最佳模型已保存！最佳验证准确率: {best_val_acc:.2f}%")

logger.info("\n🎉 训练完成！")
logger.info(f"最终最佳验证准确率: {best_val_acc:.2f}%")

2026-05-07 03:31:42,896 - Train - INFO - ==================================================
2026-05-07 03:31:42,898 - Train - INFO - 开始训练！
2026-05-07 03:31:42,899 - Train - INFO - ==================================================
2026-05-07 03:31:42,901 - Train - INFO - 
Epoch [1/50]
训练中:   0%|          | 0/165 [00:00<?, ?it/s]

验证中: 100%|██████████| 21/21 [00:08<00:00,  2.58it/s]
2026-05-07 03:32:50,815 - Train - INFO - 训练损失: 0.2314 | 训练准确率: 63.13%
2026-05-07 03:32:50,816 - Train - INFO - 验证损失: 0.2269 | 验证准确率: 63.41%
2026-05-07 03:32:50,817 - Train - INFO - 当前学习率: 0.00027135
2026-05-07 03:32:52,823 - Train - INFO - ✅ 最佳模型已保存！最佳验证准确率: 63.41%
2026-05-07 03:32:52,826 - Train - INFO - 
Epoch [2/50]
验证中: 100%|██████████| 21/21 [00:07<00:00,  2.65it/s]
2026-05-07 03:34:00,974 - Train - INFO - 训练损失: 0.2303 | 训练准确率: 63.88%
2026-05-07 03:34:00,976 - Train - INFO - 验证损失: 0.2266 | 验证准确率: 63.41%
2026-05-07 03:34:00,977 - Train - INFO - 当前学习率: 0.00019635
2026-05-07 03:34:00,978 - Train - INFO - 
Epoch [3/50]
验证中: 100%|██████████| 21/21 [00:08<00:00,  2.62it/s]
2026-05-07 03:35:10,674 - Train - INFO - 训练损失: 0.2297 | 训练准确率: 63.88%
2026-05-07 03:35:10,675 - Train - INFO - 验证损失: 0.2265 | 验证准确率: 63.41%
2026-05-07 03:35:10,676 - Train - INFO - 当前学习率: 0.00010365
2026-05-07 03:35:10,677 - Train - INFO - 
Epoch [4/50]
验证中: 100%|██